# Data Version Control

In [20]:
# Import required libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import os
import urllib.request
import zipfile

## 1. Download and Load Data

In [21]:
def load_data(file_path):
    """
    Load SMS spam data from a given file path.
    
    Parameters:
    -----------
    file_path : str
        Path to the SMS spam collection file
        
    Returns:
    --------
    pd.DataFrame
        DataFrame with 'label' and 'message' columns
    """
    # Load the data - the file is tab-separated with no header
    df = pd.read_csv(file_path, sep='\t', header=None, names=['label', 'message'])
    
    print(f"Loaded {len(df)} messages")
    print(f"Spam messages: {(df['label'] == 'spam').sum()}")
    print(f"Ham messages: {(df['label'] == 'ham').sum()}")
    
    return df

In [22]:
# Download the SMS Spam Collection dataset
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip"
zip_path = "smsspamcollection.zip"
extract_path = "sms_data"

# Download if not already present
if not os.path.exists(zip_path):
    print("Downloading SMS Spam Collection dataset...")
    urllib.request.urlretrieve(url, zip_path)
    print("Download complete!")

# Extract the zip file
if not os.path.exists(extract_path):
    print("Extracting files...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("Extraction complete!")

# The main data file
data_file = os.path.join(extract_path, "SMSSpamCollection")
print(f"Data file location: {data_file}")

Data file location: sms_data/SMSSpamCollection


In [23]:
# Load the dataset
df = load_data(data_file)
df.head(10)

Loaded 5572 messages
Spam messages: 747
Ham messages: 4825


,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
5,spam,FreeMsg Hey there darling it's been 3 week's n...
6,ham,Even my brother is not like to speak with me. ...
7,ham,As per your request 'Melle Melle (Oru Minnamin...
8,spam,WINNER!! As a valued network customer you have...
9,spam,Had your mobile 11 months or more? U R entitle...


## 2. Preprocess Data

In [24]:
def preprocess_data(df):
    df_processed = df.copy()
    
    # Remove any duplicates
    initial_count = len(df_processed)
    df_processed = df_processed.drop_duplicates()
    print(f"Removed {initial_count - len(df_processed)} duplicate messages")
    
    # Remove any missing values
    df_processed = df_processed.dropna()
    
    # Convert labels to binary (0 for ham, 1 for spam)
    df_processed['label_binary'] = (df_processed['label'] == 'spam').astype(int)
    
    # Basic text cleaning - strip whitespace
    df_processed['message'] = df_processed['message'].str.strip()
    
    # Remove any empty messages
    df_processed = df_processed[df_processed['message'].str.len() > 0]
    
    print(f"Final dataset size: {len(df_processed)} messages")
    print(f"Class distribution:")
    print(df_processed['label'].value_counts())
    
    return df_processed

In [25]:
# Preprocess the data
df_clean = preprocess_data(df)
df_clean.head()

Removed 403 duplicate messages
Final dataset size: 5169 messages
Class distribution:
label
ham     4516
spam     653
Name: count, dtype: int64


,label,message,label_binary
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0


## 2a. Save Raw Data

In [26]:
# Save the raw cleaned data to raw_data.csv for version control
df_clean.to_csv('raw_data.csv', index=False)
print("Raw data saved to raw_data.csv")
print(f"Raw data shape: {df_clean.shape}")
print(f"Class distribution in raw data:")
print(df_clean['label_binary'].value_counts())

Raw data saved to raw_data.csv
Raw data shape: (5169, 3)
Class distribution in raw data:
label_binary
0    4516
1     653
Name: count, dtype: int64


## 3. Split Data into Train/Validation/Test

In [28]:
# Split the data into training and testing sets
# Split the data into testing and validation sets

train_df, test_df = train_test_split(df_clean, test_size=0.2, random_state=42)
print(f"Training set size: {len(train_df)} messages")

val_df, test_df = train_test_split(test_df, test_size=0.5, random_state=42)

print(f"Validation set size: {len(val_df)} messages")
print(f"Test set size: {len(test_df)} messages")

Training set size: 4135 messages
Validation set size: 517 messages
Test set size: 517 messages


## 4. Save Splits to CSV Files

In [29]:
# Save the splits to CSV files
train_df.to_csv('train.csv', index=False)
val_df.to_csv('validation.csv', index=False)
test_df.to_csv('test.csv', index=False)

print("Data splits saved successfully!")
print("  - train.csv")
print("  - validation.csv")
print("  - test.csv")

Data splits saved successfully!
  - train.csv
  - validation.csv
  - test.csv


# 5. Track CSV Files with DVC

In [30]:
import subprocess
import os

# subprocess.run(['dvc', 'init', '--no-scm'], cwd=os.getcwd())

# Add raw data and split files to DVC for version control
try:
    subprocess.run(['dvc', 'add', 'raw_data.csv', 'train.csv', 'validation.csv', 'test.csv'], 
                   cwd=os.getcwd(), check=True, capture_output=True)
    print("✓ Files tracked with DVC successfully!")
except subprocess.CalledProcessError as e:
    print(f"Note: DVC files already tracked or error: {e}")

✓ Files tracked with DVC successfully!


In [31]:
# Check DVC status
try:
    result = subprocess.run(['dvc', 'status'], cwd=os.getcwd(), capture_output=True, text=True, check=True)
    if result.stdout:
        print(result.stdout)
    else:
        print("✓ All DVC files are up to date!")
except subprocess.CalledProcessError as e:
    print(f"DVC status output:\n{e.stdout if e.stdout else 'No changes'}")

Data and pipelines are up to date.



In [32]:
# Distribution of 1s and 0s in the training set, validation set, and test set

print("Training set class distribution:")
print(train_df['label_binary'].value_counts(normalize=True))

print("Validation set class distribution:")
print(val_df['label_binary'].value_counts(normalize=True))

print("Test set class distribution:")
print(test_df['label_binary'].value_counts(normalize=True))

Training set class distribution:
label_binary
0    0.875937
1    0.124063
Name: proportion, dtype: float64
Validation set class distribution:
label_binary
0    0.864603
1    0.135397
Name: proportion, dtype: float64
Test set class distribution:
label_binary
0    0.864603
1    0.135397
Name: proportion, dtype: float64


## 6. Commit Version 1 (Initial Split with random_state=42)

In [ ]:
# # Commit the current version (v1) using git
# try:
#     subprocess.run(['git', 'add', '.'], cwd=os.getcwd(), check=True, capture_output=True)
#     result = subprocess.run(['git', 'commit', '-m', 'Data v1: Initial split with random_state=42'], 
#                           cwd=os.getcwd(), capture_output=True, text=True)
#     print(f"✓ Version v1 committed with random_state=42")
#     print(f"  Message: Data v1: Initial split with random_state=42")
# except subprocess.CalledProcessError as e:
#     print(f"Commit result: {e.stdout if e.stdout else 'No changes to commit'}")

✓ Version v1 committed with random_state=42
  Message: Data v1: Initial split with random_state=42


# 5. Update CSV Files with DVC

In [34]:
# Split the data into training and testing sets
# Split the data into testing and validation sets

train_df, test_df = train_test_split(df_clean, test_size=0.2, random_state=8)
print(f"Training set size: {len(train_df)} messages")

val_df, test_df = train_test_split(test_df, test_size=0.5, random_state=8)

print(f"Validation set size: {len(val_df)} messages")
print(f"Test set size: {len(test_df)} messages")

# Save the splits to CSV files
train_df.to_csv('train.csv', index=False)
val_df.to_csv('validation.csv', index=False)
test_df.to_csv('test.csv', index=False)

print("Data splits saved successfully!")
print("  - train.csv")
print("  - validation.csv")
print("  - test.csv")

Training set size: 4135 messages
Validation set size: 517 messages
Test set size: 517 messages
Data splits saved successfully!
  - train.csv
  - validation.csv
  - test.csv


In [35]:
# Distribution of 1s and 0s in the training set, validation set, and test set

print("Training set class distribution:")
print(train_df['label_binary'].value_counts(normalize=True))

print("Validation set class distribution:")
print(val_df['label_binary'].value_counts(normalize=True))

print("Test set class distribution:")
print(test_df['label_binary'].value_counts(normalize=True))

Training set class distribution:
label_binary
0    0.874002
1    0.125998
Name: proportion, dtype: float64
Validation set class distribution:
label_binary
0    0.887814
1    0.112186
Name: proportion, dtype: float64
Test set class distribution:
label_binary
0    0.856867
1    0.143133
Name: proportion, dtype: float64


## 7. Commit Version 2 (Updated Split with random_state=8) and Demonstrate Version Control

In [ ]:
# # Commit the updated version (v2) using git
# try:
#     subprocess.run(['git', 'add', '.'], cwd=os.getcwd(), check=True, capture_output=True)
#     result = subprocess.run(['git', 'commit', '-m', 'Data v2: Updated split with random_state=8'], 
#                           cwd=os.getcwd(), capture_output=True, text=True)
#     print(f"✓ Version v2 committed with random_state=8")
#     print(f"  Message: Data v2: Updated split with random_state=8")
#     print(f"\nNotice the change in class distribution between v1 and v2!")
# except subprocess.CalledProcessError as e:
#     print(f"Commit result: {e.stdout if e.stdout else 'No changes to commit'}")

✓ Version v2 committed with random_state=8
  Message: Data v2: Updated split with random_state=8

Notice the change in class distribution between v1 and v2!
